In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score
from sklearn.calibration import CalibratedClassifierCV # 캘리브레이션 클래스
import os

# 2. 데이터 준비 및 분리
try:
    # 데이터 파일이 현재 디렉토리에 있다고 가정
    df_train = pd.read_csv("train.csv")
    df_test = pd.read_csv("test.csv")
except FileNotFoundError:
    print("오류: 데이터 파일(train.csv, test.csv)을 찾을 수 없습니다. 경로를 확인해주세요.")
    exit()

# Capping을 적용하지 않은 원본 데이터를 최종 데이터로 사용
df_train_final = df_train.copy()
df_test_final = df_test.copy()

X = df_train_final.drop(['ID', 'label'], axis=1, errors='ignore')
y = df_train_final['label']

# 학습 및 검증 데이터 분리
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# 3. 최종 최고 성능 모델 정의 및 학습 (Random Forest + Calibration)
final_base_params = {
    'n_estimators': 314,
    'max_depth': 13,
    'min_samples_split': 5,
    'min_samples_leaf': 2,
    'max_features': 'sqrt',
    'random_state': 42,
    'n_jobs': -1 # CPU 코어 전체 사용
}
base_rf_clf = RandomForestClassifier(**final_base_params)

# 모델 캘리브레이션 적용 (Isotonic)
# 이진 분류에서 확률 예측의 정확도를 높여 ROC-AUC 성능 향상에 도움을 줄 수 있음
calibrated_clf = CalibratedClassifierCV(
    estimator=base_rf_clf,
    method='isotonic',
    cv=5, # 5-fold cross-validation
    n_jobs=-1
)

print("--- 최종 최고 성능 Random Forest 모델 학습 중 (Isotonic Calibration 적용) ---")
calibrated_clf.fit(X_train, y_train)


# 4. 성능 확인
print("\n--- 캘리브레이션된 모델의 임계값(Threshold) 최적화 ---")
# 검증 데이터셋에 대한 클래스 1(양성) 확률 예측
y_proba_local = calibrated_clf.predict_proba(X_test)[:, 1]

# 정확도를 최대화하는 최적의 임계값 찾기
thresholds = np.arange(0.30, 0.70, 0.01)
best_accuracy = 0
optimal_threshold_found = 0.5

for threshold in thresholds:
    y_pred_threshold = (y_proba_local >= threshold).astype(int)
    current_accuracy = accuracy_score(y_test, y_pred_threshold)

    if current_accuracy > best_accuracy:
        best_accuracy = current_accuracy
        optimal_threshold_found = threshold

# 최적 임계값을 적용한 최종 예측
y_pred_final = (y_proba_local >= optimal_threshold_found).astype(int)

final_acc = accuracy_score(y_test, y_pred_final)
final_f1 = f1_score(y_test, y_pred_final)
# 오류 수정: 정의된 y_proba_local 변수를 사용하도록 수정
final_auc = roc_auc_score(y_test, y_proba_local)

# ===============================
# 5. 최종 결과 출력
# ===============================

print("\n==============================================")
print("✨ === 최종 최고 성능 모델 (Capping 제외 버전) ===")
print(f"최종 확정된 최적 임계값: {optimal_threshold_found:.2f}")
print("----------------------------------------------")
print(f"최종 AUC-ROC: {final_auc:.4f}")
print(f"최종 Accuracy: {final_acc:.4f}")
print(f"최종 F1-Score: {final_f1:.4f}")
print("==============================================")

--- 최종 최고 성능 Random Forest 모델 학습 중 (Isotonic Calibration 적용) ---

--- 캘리브레이션된 모델의 임계값(Threshold) 최적화 ---

✨ === 최종 최고 성능 모델 (Capping 제외 버전) ===
최종 확정된 최적 임계값: 0.51
----------------------------------------------
최종 AUC-ROC: 0.8296
최종 Accuracy: 0.7593
최종 F1-Score: 0.6848


In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
# 필요한 모든 지표 import: precision_score, recall_score, classification_report 추가
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, precision_score, recall_score, classification_report
from sklearn.calibration import CalibratedClassifierCV # 캘리브레이션 클래스
import os

# 1. (삭제됨) 사용자 정의 Capping 함수 및 규칙 정의 - Capping 제외 요청에 따라 이 섹션은 삭제되었습니다.

# 2. 데이터 준비 및 분리
try:
    # 데이터 파일이 현재 디렉토리에 있다고 가정
    df_train = pd.read_csv("train.csv")
    df_test = pd.read_csv("test.csv")
except FileNotFoundError:
    print("오류: 데이터 파일(train.csv, test.csv)을 찾을 수 없습니다. 경로를 확인해주세요.")
    exit()

# Capping을 적용하지 않은 원본 데이터를 최종 데이터로 사용
df_train_final = df_train.copy()
df_test_final = df_test.copy()

X = df_train_final.drop(['ID', 'label'], axis=1, errors='ignore')
y = df_train_final['label']

# 학습 및 검증 데이터 분리
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# 3. 최종 최고 성능 모델 정의 및 학습 (Random Forest + Calibration)
final_base_params = {
    'n_estimators': 314,
    'max_depth': 13,
    'min_samples_split': 5,
    'min_samples_leaf': 2,
    'max_features': 'sqrt',
    'random_state': 42,
    'n_jobs': -1 # CPU 코어 전체 사용
}
base_rf_clf = RandomForestClassifier(**final_base_params)

# 모델 캘리브레이션 적용 (Isotonic)
calibrated_clf = CalibratedClassifierCV(
    estimator=base_rf_clf,
    method='isotonic',
    cv=5, # 5-fold cross-validation
    n_jobs=-1
)

print("--- 최종 최고 성능 Random Forest 모델 학습 중 (Isotonic Calibration 적용) ---")
calibrated_clf.fit(X_train, y_train)


# 4. 성능 확인
print("\n--- 캘리브레이션된 모델의 임계값(Threshold) 최적화 ---")
# 검증 데이터셋에 대한 클래스 1(양성) 확률 예측
y_proba_local = calibrated_clf.predict_proba(X_test)[:, 1]

# 정확도를 최대화하는 최적의 임계값 찾기
thresholds = np.arange(0.30, 0.70, 0.01)
best_accuracy = 0
optimal_threshold_found = 0.5

for threshold in thresholds:
    y_pred_threshold = (y_proba_local >= threshold).astype(int)
    current_accuracy = accuracy_score(y_test, y_pred_threshold)

    if current_accuracy > best_accuracy:
        best_accuracy = current_accuracy
        optimal_threshold_found = threshold

# 최적 임계값을 적용한 최종 예측
y_pred_final = (y_proba_local >= optimal_threshold_found).astype(int)

# 최종 성능 지표 계산
final_acc = accuracy_score(y_test, y_pred_final)
final_f1 = f1_score(y_test, y_pred_final)
final_auc = roc_auc_score(y_test, y_proba_local)
final_precision = precision_score(y_test, y_pred_final)
final_recall = recall_score(y_test, y_pred_final)

# ===============================
# 5. 최종 결과 출력
# ===============================

print("\n==============================================")
print("✨ === 최종 최고 성능 모델 (Capping 제외 버전) ===")
print(f"최종 확정된 최적 임계값: {optimal_threshold_found:.2f}")
print("----------------------------------------------")
print(f"최종 AUC-ROC: {final_auc:.4f}")
print(f"최종 Accuracy (검증 정확도): {final_acc:.4f}")
print(f"최종 Precision (정밀도): {final_precision:.4f}")
print(f"최종 Recall (재현율): {final_recall:.4f}")
print(f"최종 F1-Score: {final_f1:.4f}")
print("==============================================")

# 분류 보고서 (Classification Report) 출력: 클래스별 정밀도/재현율 확인
print("\n--- 전체 분류 성능 보고서 (Classification Report) ---")
print(classification_report(y_test, y_pred_final))
print("-----------------------------------------------------")


# ===============================
# 6. 외부 테스트 데이터 예측 (제출용)
# ===============================
X_submission = df_test_final.drop('ID', axis=1, errors='ignore')

# 최종 모델을 사용하여 외부 테스트 데이터에 대한 예측 확률 계산
y_proba_submission = calibrated_clf.predict_proba(X_submission)[:, 1]

print(f"\n외부 테스트 데이터에 대한 예측 확률 (제출용) 할당 완료 (길이: {len(y_proba_submission)}).")

submission = pd.read_csv("sample_submission.csv")

# ROC-AUC를 평가 지표로 사용하므로 확률 값을 제출합니다.
submission["Outcome"] = y_proba_submission

# 제출 파일 저장
# submission_file_name = "submission_no_capping.csv"
# submission.to_csv(submission_file_name, index=False)
# print(f"제출 파일 저장 완료! 파일명: {submission_file_name} 👍")

--- 최종 최고 성능 Random Forest 모델 학습 중 (Isotonic Calibration 적용) ---

--- 캘리브레이션된 모델의 임계값(Threshold) 최적화 ---

✨ === 최종 최고 성능 모델 (Capping 제외 버전) ===
최종 확정된 최적 임계값: 0.51
----------------------------------------------
최종 AUC-ROC: 0.8296
최종 Accuracy (검증 정확도): 0.7593
최종 Precision (정밀도): 0.6595
최종 Recall (재현율): 0.7121
최종 F1-Score: 0.6848

--- 전체 분류 성능 보고서 (Classification Report) ---
              precision    recall  f1-score   support

           0       0.82      0.79      0.81       886
           1       0.66      0.71      0.68       514

    accuracy                           0.76      1400
   macro avg       0.74      0.75      0.75      1400
weighted avg       0.76      0.76      0.76      1400

-----------------------------------------------------

외부 테스트 데이터에 대한 예측 확률 (제출용) 할당 완료 (길이: 3000).
